# Week 3c: Quantum Kernel Results — CIFAR-10 CLIP (Dataset 3, N=1000)
**Project**: Quantum Kernels on CLIP/BERT Embeddings — ICLR Extension

**What this notebook does:**
1. Loads CIFAR-10 CLIP embeddings from Week 2c
2. Computes 1000x1000 quantum kernel matrix (IQP, 8 qubits)
3. Compares quantum kernel SVM against classical baselines
4. Generates learning curves and the full 3-dataset comparison
5. Saves everything immediately to Drive

**This is the vision modality** — airplane vs automobile, CLIP encoder, binary F1.
Completes the cross-modality study: BERT text (SST-2, AG News) + CLIP vision (CIFAR-10).

**Runtime:** ~83 min (same N=1000 kernel matrix)

In [ ]:
!pip install pennylane pennylane-lightning --quiet

import pennylane as qml
import numpy as np
import time, json, pickle, os, shutil, warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
warnings.filterwarnings('ignore')

print(f"PennyLane {qml.__version__}")
DEVICE = qml.device('lightning.qubit', wires=8)

---
## Step 0: Storage + checkpoint helpers

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

def _persist_and_download(path):
    if os.path.abspath(SAVE_DIR) != os.getcwd():
        shutil.copy(path, os.path.join(SAVE_DIR, os.path.basename(path)))
        print(f"  saved -> {SAVE_DIR}/{os.path.basename(path)}")
    try:
        from google.colab import files
        files.download(path)
        print(f"  download -> {os.path.basename(path)}")
    except Exception as e:
        print(f"  (download skipped: {type(e).__name__} — saved above)")

def checkpoint(filename, save_fn):
    save_fn(filename); _persist_and_download(filename)

def save_figure(fig, name):
    for ext in ('png', 'pdf'):
        p = f"{name}.{ext}"
        fig.savefig(p, bbox_inches='tight', dpi=150)
        _persist_and_download(p)

def _find(fname):
    dp = os.path.join(SAVE_DIR, fname)
    return dp if os.path.exists(dp) else fname

print("Helpers ready.")

---
## Step 1: Load CIFAR-10 CLIP data

In [ ]:
data = np.load(_find('cifar10_embeddings.npz'))
X_tr = data['X_tr_subset'].astype(np.float64)   # (1000, 8)
y_tr = data['y_tr_subset']
X_te = data['X_te_subset'].astype(np.float64)   # (200, 8)
y_te = data['y_te_subset']

CIFAR_CLASSES = {0: 'airplane', 1: 'automobile'}

with open(_find('cifar10_baseline_results.pkl'), 'rb') as f:
    baselines = pickle.load(f)

print(f"Train {X_tr.shape}  Test {X_te.shape}")
print(f"Value range: [{X_tr.min():.4f}, {X_tr.max():.4f}]  (must be in [0, pi])")
assert X_tr.shape == (1000, 8), "Expected (1000, 8) — re-run Week 2c"
assert X_te.shape == (200, 8),  "Expected (200, 8)  — re-run Week 2c"
print("N=1000 confirmed. Vision modality (CLIP embeddings).")

print(f"\nClass balance: train {np.mean(y_tr==1):.1%} automobile | test {np.mean(y_te==1):.1%}")
print(f"\nClassical baselines (CLIP, from Week 2c):")
print(f"  {'Kernel':<14} {'Accuracy':>10} {'F1':>8}")
print("-"*36)
for k, v in baselines.items():
    print(f"  {k:<14} {v['accuracy']:>10.4f} {v['f1']:>8.4f}")

---
## Step 2: IQP quantum kernel (identical circuit to Week 3 / 3b)

In [ ]:
N_QUBITS = 8

def iqp_feature_map(x, wires):
    n = len(wires)
    for w in wires: qml.Hadamard(wires=w)
    for i, w in enumerate(wires): qml.RZ(x[i], wires=w)
    for i in range(n - 1):
        qml.CNOT(wires=[wires[i], wires[i+1]])
        qml.RZ(x[i] * x[i+1], wires=wires[i+1])
        qml.CNOT(wires=[wires[i], wires[i+1]])

@qml.qnode(DEVICE)
def kernel_circuit(x1, x2):
    iqp_feature_map(x1, wires=range(N_QUBITS))
    qml.adjoint(iqp_feature_map)(x2, wires=range(N_QUBITS))
    return qml.probs(wires=range(N_QUBITS))

quantum_kernel = lambda a, b: float(kernel_circuit(a, b)[0])

assert np.isclose(quantum_kernel(X_tr[0], X_tr[0]), 1.0, atol=1e-5), "Kernel broken"
t0 = time.time()
for _ in range(3): quantum_kernel(X_tr[0], X_tr[1])
ms = (time.time()-t0)/3*1000
print(f"Sanity OK. Single eval {ms:.1f}ms")
print(f"N=1000 matrix estimate: ~{1000*1000*ms/1000/60:.0f} min")

---
## Step 3: Compute kernel matrix — saved immediately after (~83 min)

In [ ]:
print(f"Computing {N_QUBITS}-qubit IQP kernel matrix (N=1000, CLIP embeddings)...")
print("Saving immediately when done. Do not close tab.")
t0 = time.time()
K_train = qml.kernels.kernel_matrix(X_tr, X_tr, quantum_kernel)
K_test  = qml.kernels.kernel_matrix(X_te, X_tr, quantum_kernel)
elapsed = time.time() - t0
print(f"\nDone in {elapsed/60:.1f} min")

print("\nSaving kernel matrices immediately:")
checkpoint('cifar10_kernel_matrices.npz',
           lambda p: np.savez(p, K_train_quantum=K_train,
                              K_test_quantum=K_test, X_tr=X_tr, y_tr=y_tr))
print("\nSafe. Kernel matrices preserved.")

---
## Step 4: Quantum SVM + comparison

In [ ]:
qsvm   = SVC(kernel='precomputed', C=1.0)
qsvm.fit(K_train, y_tr)
y_pred = qsvm.predict(K_test)

acc_q = float(accuracy_score(y_te, y_pred))
f1_q  = float(f1_score(y_te, y_pred))   # binary F1 — only 2 classes

print(f"{'='*52}")
print(f"QUANTUM KERNEL SVM (N=1000, 8 qubits, IQP, CLIP):")
print(f"  Accuracy : {acc_q:.4f}")
print(f"  F1 score : {f1_q:.4f}")
print(f"{'='*52}\n")

print(f"vs classical baselines (CLIP, N=1000):")
for name, vals in baselines.items():
    diff_acc = acc_q - vals['accuracy']
    diff_f1  = f1_q  - vals['f1']
    print(f"  vs {name:<12}: acc {diff_acc:+.4f}  F1 {diff_f1:+.4f}")

---
## Step 5: Results table (all N=1000)

In [ ]:
results_all = {name: {'accuracy': v['accuracy'], 'f1': v['f1']}
               for name, v in baselines.items()}
results_all['Quantum (IQP)'] = {'accuracy': acc_q, 'f1': f1_q}

print(f"{'Method':<16} {'Accuracy':>10} {'F1':>8}")
print("-"*36)
for name, v in results_all.items():
    marker = " <-- best F1" if v['f1'] == max(r['f1'] for r in results_all.values()) else ""
    print(f"{name:<16} {v['accuracy']:>10.4f} {v['f1']:>8.4f}{marker}")

---
## Step 6: Comparison figure

In [ ]:
names  = list(results_all.keys())
accs   = [results_all[n]['accuracy'] for n in names]
f1s    = [results_all[n]['f1'] for n in names]
colors = ['#5B8DB8', '#5B8DB8', '#5B8DB8', '#D95F02']

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, vals, ylabel, title in [
    (axes[0], accs, 'Accuracy', 'Accuracy — CIFAR-10 CLIP N=1000'),
    (axes[1], f1s,  'F1',       'F1 — CIFAR-10 CLIP N=1000')]:
    ax.bar(names, vals, color=colors, edgecolor='white', linewidth=1.5, width=0.6)
    ax.set_ylim(0.4, 1.0); ax.set_title(title); ax.set_ylabel(ylabel)
    for i, v in enumerate(vals):
        ax.text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=11, fontweight='500')
    ax.set_xticklabels(names, rotation=15, ha='right')

blue   = mpatches.Patch(color='#5B8DB8', label='Classical kernels')
orange = mpatches.Patch(color='#D95F02', label='Quantum IQP (8 qubits)')
fig.legend(handles=[blue, orange], loc='upper center', ncol=2,
           fontsize=11, bbox_to_anchor=(0.5, 1.02))
plt.suptitle('Quantum vs Classical SVM — 8-dim CLIP Embeddings (CIFAR-10, vision)', y=1.06)
plt.tight_layout(); plt.show()
save_figure(fig, 'cifar10_fig1_comparison')

---
## Step 7: Learning curve

In [ ]:
train_sizes = [50, 100, 200, 400, 700, 1000]
lc = {'Quantum (IQP)': [], 'Linear': [], 'RBF': []}

print(f"{'N':>5}  {'Quantum':>9}  {'Linear':>9}  {'RBF':>9}")
print("-"*40)
for n in train_sizes:
    Kn      = K_train[:n, :n]
    Kn_test = K_test[:, :n]
    yp_q = SVC(kernel='precomputed', C=1.0).fit(Kn, y_tr[:n]).predict(Kn_test)
    lc['Quantum (IQP)'].append(float(accuracy_score(y_te, yp_q)))
    for cname, ck in [('Linear','linear'), ('RBF','rbf')]:
        clf = SVC(kernel=ck, C=1.0, gamma='scale' if ck=='rbf' else 1.0,
                  random_state=42).fit(X_tr[:n], y_tr[:n])
        lc[cname].append(float(accuracy_score(y_te, clf.predict(X_te))))
    print(f"{n:>5}  {lc['Quantum (IQP)'][-1]:>9.4f}  "
          f"{lc['Linear'][-1]:>9.4f}  {lc['RBF'][-1]:>9.4f}")

fig2 = plt.figure(figsize=(8, 4))
cmap = {'Quantum (IQP)':'#D95F02','Linear':'#5B8DB8','RBF':'#2D9C4A'}
for m, vals in lc.items():
    ls = '-' if m=='Quantum (IQP)' else '--'
    lw = 2.5 if m=='Quantum (IQP)' else 1.8
    plt.plot(train_sizes, vals, label=m, color=cmap[m],
             linestyle=ls, linewidth=lw, marker='o', markersize=5)
plt.xlabel('Training set size (N)'); plt.ylabel('Test accuracy')
plt.title('Learning curves — CIFAR-10 CLIP (vision)\n8-dim embeddings, airplane vs automobile')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.show()
save_figure(fig2, 'cifar10_fig2_learning_curves')

---
## Step 8: Full 3-dataset comparison (the paper's headline figure)

In [ ]:
# Load all three datasets' quantum + classical results
sst2, agnews = None, None
try:
    with open(_find('results_week3.json')) as f: sst2 = json.load(f)
except: pass
try:
    with open(_find('ag_news_results_week3b.json')) as f: agnews = json.load(f)
except: pass

if sst2 and agnews:
    methods = ['Linear', 'RBF', 'Polynomial', 'Quantum (IQP)']

    # accuracy across 3 datasets
    sst2_acc   = [sst2['all_results'][m]['accuracy'] for m in methods]
    agnews_acc = [agnews['all_results'][m]['accuracy'] for m in methods]
    cifar_acc  = [results_all[m]['accuracy'] for m in methods]

    fig3, ax = plt.subplots(figsize=(13, 6))
    x = np.arange(len(methods)); w = 0.25
    bars1 = ax.bar(x-w, sst2_acc,   w, label='SST-2 (BERT, sentiment)',  color='#5B8DB8', edgecolor='white')
    bars2 = ax.bar(x,   agnews_acc, w, label='AG News (BERT, topic)',    color='#2D9C4A', edgecolor='white')
    bars3 = ax.bar(x+w, cifar_acc,  w, label='CIFAR-10 (CLIP, vision)',  color='#D95F02', edgecolor='white')

    for bars in [bars1, bars2, bars3]:
        for b in bars:
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
                    f'{b.get_height():.2f}', ha='center', fontsize=8)

    ax.set_xticks(x); ax.set_xticklabels(methods, fontsize=11)
    ax.set_ylim(0.4, 0.9); ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title('Quantum vs Classical Kernels Across Three Datasets and Two Modalities (N=1000)',
                 fontsize=13)
    ax.legend(fontsize=11, loc='upper right')
    ax.axvspan(2.625, 3.375, alpha=0.08, color='orange')  # highlight quantum column
    plt.tight_layout(); plt.show()
    save_figure(fig3, 'THREE_DATASET_headline_comparison')
    print("Headline 3-dataset figure saved.")

    # Print the master table
    print("\n" + "="*64)
    print("MASTER RESULTS TABLE — 3 datasets, N=1000")
    print("="*64)
    print(f"{'Method':<16} {'SST-2':>10} {'AG News':>10} {'CIFAR-10':>10}")
    print(f"{'':16} {'(acc)':>10} {'(acc)':>10} {'(acc)':>10}")
    print("-"*64)
    for m in methods:
        print(f"{m:<16} {sst2['all_results'][m]['accuracy']:>10.4f} "
              f"{agnews['all_results'][m]['accuracy']:>10.4f} "
              f"{results_all[m]['accuracy']:>10.4f}")
else:
    missing = []
    if not sst2: missing.append('results_week3.json (SST-2)')
    if not agnews: missing.append('ag_news_results_week3b.json (AG News)')
    print(f"Cannot build 3-dataset figure. Missing from Drive: {missing}")
    print("Upload them and re-run this cell.")

---
## Step 9: Save results

In [ ]:
results_week3c = {
    'dataset': 'cifar10_binary',
    'modality': 'vision',
    'encoder': 'openai/clip-vit-base-patch32',
    'classes': CIFAR_CLASSES,
    'n_qubits': N_QUBITS,
    'n_train': int(len(X_tr)),
    'n_test': int(len(X_te)),
    'feature_map': 'IQP (Hadamard + RZ + ZZ)',
    'compute_time_min': round(elapsed/60, 1),
    'quantum_accuracy': acc_q,
    'quantum_f1': f1_q,
    'all_results': results_all,
    'learning_curve_sizes': train_sizes,
    'learning_curve_results': {k: v for k, v in lc.items()},
}
checkpoint('cifar10_results_week3c.json',
           lambda p: json.dump(results_week3c, open(p,'w'), indent=2))

print("\n=== WEEK 3c DONE ===")
print(f"Dataset       : CIFAR-10 binary (vision, CLIP)")
print(f"Quantum IQP   : acc={acc_q:.4f}  F1={f1_q:.4f}")
print(f"Best classical: acc={max(v['accuracy'] for v in baselines.values()):.4f}")
print(f"\nThree-dataset study complete: SST-2, AG News, CIFAR-10")

---
## Week 3c complete — Three-Dataset Study Done

### Saved + downloaded:
- `cifar10_kernel_matrices.npz` — saved immediately after the 83-min compute
- `cifar10_fig1_comparison.png/.pdf` — quantum vs classical bars
- `cifar10_fig2_learning_curves.png/.pdf`
- `THREE_DATASET_headline_comparison.png/.pdf` — the paper's main figure
- `cifar10_results_week3c.json`

### You now have the complete cross-modality picture:
- SST-2: text, sentiment, binary, BERT
- AG News: text, topic, 4-class, BERT
- CIFAR-10: vision, objects, binary, CLIP

### The key cross-modality question for the paper:
Does the quantum-vs-classical pattern hold across modalities? If quantum kernels
behave similarly on CLIP vision embeddings as on BERT text embeddings (competitive
at large N, weaker at small N), that is a strong generalisation claim. If they
behave differently, that itself is an interesting finding about how embedding
geometry differs across modalities.

### Remaining work:
- Week 4c: ablations on CIFAR-10 (optional — you have ablations on 2 datasets already)
- Week 5: full analysis — KTA across all 3 datasets, t-SNE, significance tests
- Week 6: paper writing

You are now in strong shape for the ICLR submission.